In [ ]:
#-------------------------------------------------------------------------------
# Name:        04_Download_GEE_images
# Purpose:     Download the S2 clip under the grid
#
# Authors:      Gijs van den Dool & Meggison Oritsemisan
#
# Created:     22/04/2024
# Copyright:   (c) Project Canopy Watch 2024
# Licence:     <Project Canopy Watch>
#-------------------------------------------------------------------------------
# Project Data Folder:
# https://drive.google.com/drive/folders/1Xjhe6JXYHi8U_q-eseQ1nx4FUVBc-1Tk

# Vector Masks:
# Output from the selection of IA areas in a 2560x2560 bounding box, retaining
# only the bounding box

# Raster Masks:
# Input for the modelling, as a input for predictions

# Disclaimer: most code was developed by caleb, to work on the whole tile, the
# modification is that it performs the clip on the bounding area of the grid
# and exports in a 256x256 patch


In [ ]:
import os

import ee
import geemap
import geemap.foliumap as emap

import datetime

import json

from shapely.geometry import Point,LineString, Polygon

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import geopandas as gpd

from matplotlib import pyplot as plt

from datetime import datetime as DT
from tqdm.notebook import tqdm_notebook

In [ ]:
try:
        ee.Initialize()
except Exception as e:
        ee.Authenticate()
        ee.Initialize()

In [ ]:
# change these to match your configuration:
project_root = "/content/drive/MyDrive/Project Canopy Official Folder/Task 02 Data Preprocessing/Working Files/Experiment_IA" # Change Project File
task_folder = "02_WorkingFiles"
file_name = "ImagePatch_IA_49.geojson"  # Corrected file name

# Construct the file path using os.path.join()
file_path = os.path.join(project_root, task_folder,file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
    print("File exists.")
else:
    print("File does not exist or is not accessible.")

File exists.


In [ ]:
# Define the file path to the shapefile
task_folder = "03_Output_Data"

# Create main output folder:
out_folder = os.path.join(project_root, task_folder, "02_download_Sentinel2_GEE/")

if os.path.exists(out_folder) == False:
  os.makedirs(out_folder)

  # Create output folder SR:
  out_folder1 = os.path.join(out_folder, "ImagePatch_SR/")
  os.makedirs(out_folder1)

  # Create output folder HM:
  out_folder2 = os.path.join(out_folder, "ImagePatch_HM/")
  os.makedirs(out_folder2)

else:
  print("folder already created")
  out_folder1 = os.path.join(out_folder, "ImagePatch_SR/")
  out_folder2 = os.path.join(out_folder, "ImagePatch_HM/")


folder already created


In [ ]:
gdf_ImagePatch = gpd.read_file(file_path)

print(len(gdf_ImagePatch)) # total lines in the AoI
print(gdf_ImagePatch.crs)  # current projection
gdf_ImagePatch.head(3)

6
EPSG:4326


,GID,geometry
0,GID_IA_49_0_1,"POLYGON ((18.67514 -4.54758, 18.67545 -4.57041..."
1,GID_IA_49_1_1,"POLYGON ((18.67545 -4.57041, 18.67514 -4.54758..."
2,GID_IA_49_1_1,"POLYGON ((18.67545 -4.57041, 18.67514 -4.54758..."


In [ ]:
working_folder = "01_Input_Data"
file_name = "IA_49.geojson" # might be different for other areas

# file_path = project_root + task_folder + woring_folder + file_name
file_path = os.path.join(project_root, working_folder, file_name)

# Test if the path is accessible
if os.path.isfile(file_path):
  gdf_ImageAoI = gpd.read_file(file_path)

  print(len(gdf_ImageAoI)) # total lines in the AoI
  print(gdf_ImageAoI.crs)  # current projection
  print(gdf_ImageAoI)
else:
  print("file missing")

1
EPSG:4326
  image_AoI    min_date    max_date  \
0     IA_49  2024-03-13  2024-03-15   

                                                 url  image_date  \
0  https://apps.sentinel-hub.com/eo-browser/?zoom...  2024-03-14   

                                            geometry  
0  POLYGON ((18.65303 -4.59464, 18.72239 -4.59487...  


In [ ]:
gdf_ImageAoI.url[0]

'https://apps.sentinel-hub.com/eo-browser/?zoom=17&lat=-4.55536&lng=18.69361&themeId=DEFAULT-THEME&visualizationUrl=https%3A%2F%2Fservices.sentinel-hub.com%2Fogc%2Fwms%2Fbd86bcc0-f318-402b-a145-015f85b9427e&datasetId=S2L2A&fromTime=2024-03-14T00%3A00%3A00.000Z&toTime=2024-03-14T23%3A59%3A59.999Z&layerId=1_TRUE_COLOR&demSource3D=%22MAPZEN%22'

In [ ]:
def test_image_date(date_min, date_max):
  booImageFound = False

  s2_sr = ee.ImageCollection('COPERNICUS/S2_SR')\
    .filterDate(date_min, date_max)\
    .filterBounds(geometry) \
    .first() \

  try:
    #print(s2_sr.getInfo())
    sys_date = ee.Date(s2_sr.get('system:time_start'))

    time = sys_date.getInfo()
    #print(time.items())
    #print(time.get("value"))

    dt_time = datetime.datetime.fromtimestamp(time.get("value") // 1000)
    #print(dt_time)
    booImageFound = True

  except:
    #print("date is not valid, increasing window")
    booImageFound = False


  return booImageFound

In [ ]:
def getEVI(image):
    # Compute the EVI using an expression.
    EVI = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),
            'BLUE': image.select('B2').divide(10000)
        }).rename("B_EVI")

    image = image.addBands(EVI)

    return(image)

#   Calculate MSAVI2
def getMSAVI2(image):
    # Compute the MSAVI2 using an expression.
    MSAVI2 = image.expression(
        '(2 * NIR + 1 - sqrt(pow((2 * NIR + 1), 2) - 8 * (NIR - RED))) / 2', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),

        }).rename("B_MSAVI2")

    image = image.addBands(MSAVI2)

    return(image)

def getNDMI(image):
    # Compute the NDMI using an expression.
    NDMI = image.expression(
        '(B8 - B11) / (B8 + B11)', {
            'B8': image.select('B8').divide(10000),
            'B11': image.select('B11').divide(10000),

        }).rename("B_NDMI")

    #alternative:
    #ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')

    image = image.addBands(NDMI)

    return(image)

def getNBR(image):
    # Compute the NBR using an expression.
    NBR = image.expression(
        '(NIR - SWIR) / (NIR + SWIR)', {
            'NIR': image.select('B8').divide(10000),
            'SWIR': image.select('B12').divide(10000),

        }).rename("B_NBR")

    #alternative:
    #nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')

    image = image.addBands(NBR)

    return(image)

#   calculated as  NDBI= (SWIR-NIR/(SWIR+NIR)
def getNDBI(image):
    # Compute the NDBI using an expression.
    NDBI = image.expression(
        '(SWIR -NIR) / (SWIR + NIR)', {
            'NIR': image.select('B8').divide(10000),
            'SWIR': image.select('B11').divide(10000),

        }).rename("B_NDBI")

    #alternative:
    #ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')

    image = image.addBands(NDBI)

    return(image)

def getNDWI(image):
    # Compute the MSAVI2 using an expression.
    NDWI = image.expression(
        '(GREEN - NIR) / (GREEN + NIR)', {
            'NIR': image.select('B8').divide(10000),
            'GREEN': image.select('B3').divide(10000),

        }).rename("B_NDWI")

    image = image.addBands(NDWI)

    #alternative:
    #ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    return(image)

#   calculated as NDVI = (NIR-RED) / (NIR+RED)

def getNDVI(image):
    # Compute the NDVI using an expression.
    NDVI = image.expression(
        '(NIR - RED) / (NIR + RED)', {
            'NIR': image.select('B8').divide(10000),
            'RED': image.select('B4').divide(10000),

        }).rename("B_NDVI")

    #alternative:
    #ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    image = image.addBands(NDVI)

    return(image)

In [ ]:
def addDate(image):
    img_date = ee.Date(image.date())
    img_date = ee.Number.parse(img_date.format('YYYYMMdd'))
    return image.addBands(ee.Image(img_date).rename('date').toInt())

In [ ]:
print("steps: ", len(gdf_ImagePatch))


steps:  6


In [ ]:
def adjust_geometry_for_fixed_size_output(centroid, target_pixels, scale):
    """
    Adjusts the geometry around a centroid to ensure the exported image matches the target pixel dimensions.

    param centroid: Tuple of (longitude, latitude) for the AOI's centroid.
    param target_pixels: The target size in pixels (e.g., 128 for a 128x128 image).
    param scale: The scale in meters per pixel.

    return: An ee.Geometry.Rectangle object for the adjusted region.

    """

    # include ajustement factor
    # adjustment_factor = 0.9949 # Adjust this factor as needed to fine-tune the output size

    # Calculate the physical size of the target area in meters
    target_size_meters = target_pixels * scale

    # Convert meters to degrees approximately (1 degree = approx. 111,319.9 meters)
    target_size_degrees = target_size_meters / 111319.9

    # Calculate the bounds of the rectangle
    half_size_degrees = target_size_degrees / 2
    min_lon = centroid[0] - half_size_degrees
    max_lon = centroid[0] + half_size_degrees
    min_lat = centroid[1] - half_size_degrees
    max_lat = centroid[1] + half_size_degrees

    # Create and return the geometry
    return ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])



In [ ]:
# Main loop
folder_SR = out_folder1 #'Omdena_CanopyWatch_personal' #to create such a folder in your drive
folder_HM = out_folder2

# for k in range(1): #range(len(gdf_ImagePatch)):
for k in tqdm_notebook(range(len(gdf_ImagePatch))):

####Generate centroid and the bounding box of the aoi
    aoi = gdf_ImagePatch.iloc[k]
    strGID = aoi.GID

    # work on the geometries:
    centroid=aoi.geometry.centroid.coords[0]
    fixed_pixels = 256  # Desired output image width in pixels
    scale = 10

    adjusted_geometry = adjust_geometry_for_fixed_size_output(
    centroid,
    target_pixels=fixed_pixels,
    scale=scale
    )

    minx=aoi.geometry.bounds[0]
    miny=aoi.geometry.bounds[1]
    maxx=aoi.geometry.bounds[2]
    maxy=aoi.geometry.bounds[3]
    geometry = ee.Geometry.Rectangle([minx,miny,maxx,maxy])

    #Find the dates:
    date_min = gdf_ImageAoI.min_date[0]
    date_max = gdf_ImageAoI.max_date[0]

    booPass = False
    booTest = True # we are presuming that we will find an image
    int_tests = 10  # limiting to 3 days around the saved date

    while booPass == False:
        for test in range(int_tests):

            booPass = test_image_date(date_min, date_max)
            #print(booPass)
            if booPass == True:
                # print("date found, end test")
                break
            else:

                # changing the dates:
                dt_object_min = DT.strptime(date_min, '%Y-%m-%d')
                dt_object_min = dt_object_min - datetime.timedelta(days=1)

                date_min = dt_object_min.strftime('%Y-%m-%d')

                dt_object_max = DT.strptime(date_max, '%Y-%m-%d')
                dt_object_max = dt_object_max + datetime.timedelta(days=1)

                date_max = dt_object_max.strftime('%Y-%m-%d')

            #end if

            # Stopping condition:
            if test == int_tests-1: # fail the test:
                booPass = True        # stop the loop
                booTest = False
            # end if
        #end for
    # end while
    #print("date range: ", date_min, " - ", date_max)

    if booTest:
    # bringing in the process design from the bog

    # filter sentinel 2 images and apply the formulas, finally we obtain
    # single image with median operation (to reduce the image if necessary)

    # Add bands to image, in this order to be consisten:
        image_S2_SR = ee.ImageCollection('COPERNICUS/S2') \
            .filterDate(date_min, date_max).filterBounds(adjusted_geometry) \
            .map(getNDVI) \
            .map(getEVI) \
            .map(getMSAVI2) \
            .map(getNDMI) \
            .map(getNBR) \
            .map(getNDBI) \
            .map(getNDWI) \
            .map(addDate).median()

        image_S2_SR = image_S2_SR.select('B.+')
        print("Number of bands after adding extra bands for SR:", image_S2_SR.bandNames().size().getInfo())

        image_S2_HM = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
            .filterDate(date_min, date_max).filterBounds(adjusted_geometry) \
            .map(getNDVI) \
            .map(getEVI) \
            .map(getMSAVI2) \
            .map(getNDMI) \
            .map(getNBR) \
            .map(getNDBI) \
            .map(getNDWI) \
            .map(addDate).median()

        image_S2_HM = image_S2_HM.select('B.+')
        print("Number of bands after adding extra bands for HM:", image_S2_HM.bandNames().size().getInfo())

        try:
            # Construct file paths for the images
            file_path_SR = os.path.join(folder_SR, f"image_{strGID}.tif")
            file_path_HM = os.path.join(folder_HM, f"image_{strGID}.tif")

            # Export images using geemap, ensure your image variables and geometry are correctly initialized
            # check if image exists
            if os.path.isfile(file_path_SR) and os.path.isfile(file_path_HM):
                print("file exists")
            else:
                geemap.ee_export_image(image_S2_SR, filename=file_path_SR, scale=10, region=adjusted_geometry.getInfo())
                geemap.ee_export_image(image_S2_HM, filename=file_path_HM, scale=10, region=adjusted_geometry.getInfo())
                print(image_S2_HM.getInfo())
                print(date_max, date_min)

        except:
            print("!!!!Issue!!!!.....check {}........!!!!!!".format(strGID))

    else:
        print("no image found")

print("Done")

  0%|          | 0/6 [00:00<?, ?it/s]

Number of bands after adding extra bands for SR: 19
Number of bands after adding extra bands for HM: 19
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_IA/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/image_GID_IA_49_0_1.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_IA/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_HM/image_GID_IA_49_0_1.tif
Number of bands after adding extra bands for SR: 19
Number of bands after adding extra bands for HM: 19
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_IA/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch_SR/image_GID_IA_49_1_1.tif
Generating URL ...
Please wait ...
Data downloaded to /Users/misanmeggison/Documents/task-04-data-preprocessing/Experiment_IA/03_Output_Data/02_download_Sentinel2_GEE/ImagePatch